# 📖 Notebook 1: Data Residency Basics

**Goal**: Understand why data must stay in specific geographic regions and how to route writes to the correct database.

## Learning Objectives

By the end of this notebook, you'll understand:
- What PII (Personally Identifiable Information) is and why it matters
- What data residency means under GDPR
- How Azure Paired Regions keep data within legal boundaries
- How to implement geo-routing — sending user data to the right regional database

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 08-enterprise/gdpr-paired-regions
docker compose up -d
```

### Visualization

- **Adminer** (Database GUI): http://localhost:8081  
  - EU-West: Server `postgres-eu-west`, User `demo`, Password `demo`, DB `gdpr_eu_west`
  - EU-North: Server `postgres-eu-north`, User `demo`, Password `demo`, DB `gdpr_eu_north`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
from datetime import datetime

# ── Connection settings for both regions ────────────────────
# In a real Azure deployment, these would be different Azure
# Database for PostgreSQL instances in different regions.
# Here we simulate with two Docker containers on different ports.

EU_WEST_CONFIG = {
    "host": "localhost",
    "port": 55433,
    "database": "gdpr_eu_west",
    "user": "demo",
    "password": "demo"
}

EU_NORTH_CONFIG = {
    "host": "localhost",
    "port": 55434,
    "database": "gdpr_eu_north",
    "user": "demo",
    "password": "demo"
}

def get_connection(region):
    """Get a database connection for the specified region."""
    config = EU_WEST_CONFIG if region == "eu-west" else EU_NORTH_CONFIG
    return psycopg2.connect(**config)

# Test both connections
for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    count = cur.fetchone()[0]
    print(f"✅ Connected to {region} — {count} users found")
    conn.close()

## 1. What is PII?

**PII** stands for **Personally Identifiable Information**. It is a useful shorthand, but note that GDPR does not use it — GDPR's term is **personal data**, defined in Article 4(1) as *any information relating to an identified or identifiable natural person*. That is deliberately broader than "PII": it covers data that identifies someone only *in combination* with other data you or someone else holds.

Examples in our schema:

| Data | Why It's PII |
|------|--------------|
| Full name | Directly identifies someone |
| Email address | Directly identifies someone |
| Phone number | Can be linked to a person |
| Date of birth | Combined with name, uniquely identifies |
| IP address | Can be traced to a household |
| Street address | Physical location of a person |
| Order history | Reveals personal preferences and habits |

Let's look at the PII we have in our database:

In [ ]:
# Show all PII columns for users in EU-West
conn = get_connection("eu-west")
cur = conn.cursor()

cur.execute("""
    SELECT id, email, full_name, phone, date_of_birth, country_code, home_region
    FROM users
    WHERE home_region = 'eu-west'
    ORDER BY id
""")

print("🇳🇱 EU-West Region Users (PII Data)")
print("=" * 100)
print(f"{'ID':<4} {'Email':<32} {'Name':<20} {'Phone':<20} {'DOB':<12} {'Country'}")
print("-" * 100)
for row in cur.fetchall():
    print(f"{row[0]:<4} {row[1]:<32} {row[2]:<20} {row[3] or 'N/A':<20} {str(row[4]):<12} {row[5]}")

print("\n⚠️  Every column above is personal data under GDPR — including 'ID',")
print("    which is a persistent identifier for one specific person.")
print("    Our design decision (not a GDPR rule): keep this data inside the EU/EEA")
print("    so we never have to justify a Chapter V transfer.")
conn.close()

## 2. What is Data Residency?

**Data residency** means the physical location where data is stored. Getting the GDPR relationship right matters here, because it is the single most-misquoted thing in this whole topic:

- GDPR **does not mandate** that personal data stay in the EU/EEA. There is no such article.
- What GDPR does is regulate **transfers to third countries** (Chapter V, Articles 44–50). A transfer needs an adequacy decision, appropriate safeguards (SCCs, BCRs), or a derogation — and post-*Schrems II*, a transfer impact assessment showing local law does not defeat those safeguards.
- Keeping data in-region is therefore an **engineering strategy to avoid Chapter V work**, not a legal requirement in its own right. It is a good strategy. It is not the law.
- Genuine *localisation* requirements do exist, but they come from **elsewhere**: national public-sector procurement rules, sector regulators, professional-secrecy law (e.g. §203 StGB for German health and legal professions), or a customer contract. Germany has no general "all data stays in Germany" statute — but a German hospital's supervisory authority may well insist on it, which amounts to the same thing for your architecture.

### Why This Matters for Cloud Computing

When you use a cloud provider like Azure, your data is stored in a **specific data center** in a **specific country**. You need to know:

1. **Where is my primary data?** (e.g., Netherlands)
2. **Where are my backups?** (e.g., Ireland — still in EU ✅)
3. **Where does my data go during failover?** (e.g., Ireland — still in EU ✅)
4. **Can support engineers in other countries see my data?** (needs legal basis)

### Azure Paired Regions Solve This

Microsoft pairs regions **within the same geography**:

```
West Europe (Netherlands) ←→ North Europe (Ireland)         [Geography: Europe]
France Central (Paris)    ←→ France South (Marseille)*      [Geography: France]
Germany West Central (FFM) ←→ Germany North (Berlin)*       [Geography: Germany]
Sweden Central            ←→ Sweden South*                  [Geography: Sweden]
Poland Central            ←→ (no pair — availability zones only)
```
<sub>* restricted-access region — you must request access.</sub>

**What it is not.** Microsoft's wording is *"To meet data residency requirements, **almost all** regions reside within the same geography as their pair."* Almost. **Brazil South is paired with South Central US** — a different geography, and asymmetrically at that. Also: *"a small number of Azure services use these region pairs"*, and *"deploying resources to a region in a pair doesn't automatically make them more resilient, nor does it provide automatic high availability, disaster recovery capabilities, or failover."*

So pairing is a helpful **default**, verified per service. It is not a boundary. The boundary comes from your region choices, Azure Policy denying non-approved locations, and contractual commitments such as the EU Data Boundary.

## 3. Geo-Routing: Sending Data to the Right Region

In a real system, when a user signs up or updates their profile, we need to route their data to the correct regional database. This is called **geo-routing**.

### How It Works

```
New User Signs Up
       │
       ▼
  ┌─────────────┐
  │ Geo-Router  │ ← Looks at user's country_code
  └──────┬──────┘
         │
    ┌────┴────┐
    ▼         ▼
 EU-West   EU-North
 NL,BE,    IE,SE,
 FR,DE     FI,DK
```

Let's build this:

In [ ]:
# ── Geo-Routing Configuration ──────────────────────────────
# Maps country codes to their assigned Azure paired region.
# In production, this would come from a configuration service
# or Azure Traffic Manager.

COUNTRY_TO_REGION = {
    # EU-West countries (primary: Netherlands)
    "NL": "eu-west",   # Netherlands
    "BE": "eu-west",   # Belgium
    "FR": "eu-west",   # France
    "DE": "eu-west",   # Germany
    "LU": "eu-west",   # Luxembourg
    "AT": "eu-west",   # Austria

    # EU-North countries (primary: Ireland)
    "IE": "eu-north",  # Ireland
    "SE": "eu-north",  # Sweden
    "FI": "eu-north",  # Finland
    "DK": "eu-north",  # Denmark
    "NO": "eu-north",  # Norway  — EEA, so inside the GDPR area
    "IS": "eu-north",  # Iceland — EEA, so inside the GDPR area
}

# Countries that are NOT in the EU/EEA but are covered by a European
# Commission *adequacy decision*, which is a different legal footing:
# a transfer there is lawful without SCCs, but it IS still a Chapter V
# transfer and has to be recorded as one.  Switzerland is the classic
# case people mis-file as "basically EU" — it is not in the EU or the
# EEA.  We keep them out of the EU regions on purpose.
ADEQUACY_COUNTRIES = {
    "CH": "Switzerland — adequacy decision, not EU/EEA",
    "GB": "United Kingdom — adequacy decision, not EU/EEA",
    "NZ": "New Zealand — adequacy decision",
}

def get_region_for_country(country_code: str) -> str:
    """
    Determines which Azure region should store data for a given country.
    This is the core of geo-routing.
    
    In Azure, this is handled by Azure Traffic Manager or Azure Front Door,
    which routes requests to the nearest compliant region.
    """
    cc = country_code.upper()
    region = COUNTRY_TO_REGION.get(cc)
    if region is None:
        if cc in ADEQUACY_COUNTRIES:
            raise ValueError(
                f"Country '{cc}' ({ADEQUACY_COUNTRIES[cc]}) is outside the EU/EEA. "
                f"Storing this user in an EU region is fine; storing them OUTSIDE it "
                f"is a Chapter V transfer that needs its own legal basis and record."
            )
        raise ValueError(
            f"Country '{cc}' not mapped to any region. "
            f"This user may need special handling (non-EU data residency)."
        )
    return region


# Sanity-check the routing table itself: every country must land in a
# region, every region must be one we actually have, and no country may
# be claimed by both.  A silent typo here would misroute real people.
_VALID_REGIONS = {"eu-west", "eu-north"}
assert set(COUNTRY_TO_REGION.values()) <= _VALID_REGIONS, \
    f"routing table points at unknown regions: {set(COUNTRY_TO_REGION.values()) - _VALID_REGIONS}"
assert not (set(COUNTRY_TO_REGION) & set(ADEQUACY_COUNTRIES)), \
    "a country cannot be both an EU/EEA region target and an adequacy country"

# Test the geo-router
test_countries = ["NL", "IE", "DE", "SE", "FR", "FI"]
print("🌍 Geo-Routing Table")
print("=" * 40)
for cc in test_countries:
    region = get_region_for_country(cc)
    print(f"  {cc} → {region}")

## 🚫 Bad → ✅ Best: Why Geo-Routing Matters

Before we celebrate the geo-router, let's see what happens **without** it.
Many small teams start by putting *all* users in one database — usually in the
cheapest region (often US-East). Let's see why that breaks GDPR.

### ❌ Bad: one hard-coded database for everyone

```python
# Everyone goes into the same database, wherever that happens to be
def create_user_BAD(email, country_code):
    conn = connect(THE_DATABASE)           # country_code is ignored
    conn.execute("INSERT INTO users ...")
```

We only have two EU containers, so the cell below writes a Swedish user into
**eu-west** rather than genuinely shipping her to Virginia. Be clear about what
that does and does not prove: it demonstrates the *routing* bug (the record
lands somewhere nobody chose, and its `home_region` label now contradicts the
user's country), which is the mechanical cause of the legal problem. It does
not itself demonstrate an unlawful transfer, because both databases here are
notionally in the EU.

Why the routing bug is the thing to catch:
1. **The same one-line bug that puts a Swede in eu-west puts her in us-east**
   the day someone adds a US region to the pool. The code has no opinion about
   geography, so geography is decided by whatever the config says today.
2. **Schrems II (2020)** — the CJEU invalidated the EU–US Privacy Shield.
   Transfers of EU personal data to the US are not automatically unlawful, but
   they need a valid Chapter V mechanism *plus* a transfer impact assessment.
   (The 2023 EU–US Data Privacy Framework restored an adequacy route, and is
   itself under challenge.) None of that analysis is possible if you cannot
   answer "where is this row?".
3. **You cannot answer the auditor's question.** Article 30 wants a record of
   *where* data is stored and transferred. A hard-coded connection string means
   the honest answer is "wherever this environment variable points".
4. **One fire = total loss** — no paired-region DR.

### ⚠️ Slightly-less-bad: One EU database, no paired region

Better, but if the single EU datacenter goes down, you're offline and your
backups (if any) might be in a non-compliant region.

### ✅ Best: Geo-routed writes + paired region DR

The `create_user_in_correct_region()` function we're about to build is the
"best" version:
- Routes writes by `country_code` to the **correct EU region**
- Logs residency for the audit trail
- The paired region (next notebook) provides DR **without** leaving the EU


In [ ]:
# ── Demo: the BAD way — ignoring the user's country ─────────
# This is the anti-pattern we want to AVOID.

def create_user_BAD(email, full_name, country_code):
    """
    Anti-pattern: always writes to eu-west, ignoring the user's country.
    This is how many teams start — one database, one region.
    It breaks down as soon as you have:
      • a German customer who legally needs data in Germany
      • a regulator asking "where exactly is Ms Svensson's phone number?"
      • a regional outage (no DR)
    """
    conn = get_connection("eu-west")   # ← hard-coded, no routing
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO users (email, full_name, country_code, home_region, consent_given)
        VALUES (%s, %s, %s, 'eu-west', TRUE)
        ON CONFLICT (email) DO NOTHING
        RETURNING id
    """, (email, full_name, country_code))
    row = cur.fetchone()
    conn.commit(); conn.close()
    return row[0] if row else None

# Pretend a Swedish user signs up — naively we put them in eu-west
bad_id = create_user_BAD("bad.ingrid@example.se", "Bad Ingrid", "SE")
print(f"❌ BAD: Swedish user stored in eu-west (id={bad_id})")

# Don't take the print statement's word for it — go and look.
# A residency claim you haven't queried is a residency claim you don't have.
def where_does_this_row_live(email):
    """Returns the set of regional databases that physically hold this email."""
    found = set()
    for region in ("eu-west", "eu-north"):
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("SELECT 1 FROM users WHERE email = %s", (email,))
        if cur.fetchone():
            found.add(region)
        conn.close()
    return found

bad_where = where_does_this_row_live("bad.ingrid@example.se")
correct_region = get_region_for_country("SE")
print(f"   Physically found in : {sorted(bad_where)}")
print(f"   Should have been in : {correct_region}")
assert bad_where == {"eu-west"}, f"expected the bad write to land in eu-west, got {bad_where}"
assert correct_region not in bad_where, (
    "the anti-pattern is supposed to MISROUTE this user — if she landed in "
    "her correct region the demo proves nothing"
)
print("   → An auditor would ask: 'Her country is SE but the row is in eu-west — why?'\n")

# ── Now compare with the ✅ BEST way (built in the next cells) ──
print("✅ BEST: use get_region_for_country() + create_user_in_correct_region()")
print(f"   A Swedish user would be routed to: {correct_region}")

# Clean up the bad insert so it does not pollute the rest of the notebook
conn = get_connection("eu-west")
conn.cursor().execute("DELETE FROM users WHERE email = 'bad.ingrid@example.se'")
conn.commit(); conn.close()
print("\n🧹 Cleaned up the bad-practice user.")


In [ ]:
def create_user_in_correct_region(email, full_name, phone, date_of_birth, country_code):
    """
    Creates a new user in the geographically correct database.
    
    This is how a GDPR-compliant system works:
    1. Determine the user's region from their country
    2. Connect to that region's database
    3. Insert the user data
    4. Log the data residency action for audit
    """
    # Step 1: Determine the correct region
    region = get_region_for_country(country_code)
    print(f"📍 User from {country_code} → routing to {region}")

    # Step 2: Connect to the correct regional database
    conn = get_connection(region)
    cur = conn.cursor()

    try:
        # Step 3: Insert the user
        cur.execute("""
            INSERT INTO users (email, full_name, phone, date_of_birth, country_code, home_region, consent_given, consent_date)
            VALUES (%s, %s, %s, %s, %s, %s, TRUE, NOW())
            RETURNING id
        """, (email, full_name, phone, date_of_birth, country_code, region))
        user_id = cur.fetchone()[0]

        # Step 4: Log the data residency action
        cur.execute("""
            INSERT INTO data_residency_log (user_id, action, source_region, table_name, record_id, reason)
            VALUES (%s, 'write', %s, 'users', %s, 'New user registration — geo-routed by country code')
        """, (user_id, region, user_id))

        # Also log consent
        cur.execute("""
            INSERT INTO consent_log (user_id, action, purpose, ip_address)
            VALUES (%s, 'granted', 'essential', '127.0.0.1')
        """, (user_id,))

        conn.commit()
        print(f"✅ User '{full_name}' created with ID {user_id} in {region}")
        print(f"   📋 Data residency logged | Consent recorded")
        return user_id, region

    except Exception as e:
        conn.rollback()
        print(f"❌ Error: {e}")
        raise
    finally:
        conn.close()


# ── Demo: Create users from different countries ──────────────
print("\n🆕 Creating New Users with Geo-Routing")
print("=" * 50)

# A German user → should go to EU-West (Netherlands data center)
create_user_in_correct_region(
    email="max.weber@example.de",
    full_name="Max Weber",
    phone="+49-30-5555-1234",
    date_of_birth="1991-04-20",
    country_code="DE"
)

print()

# A Swedish user → should go to EU-North (Ireland data center)
create_user_in_correct_region(
    email="ingrid.svensson@example.se",
    full_name="Ingrid Svensson",
    phone="+46-8-555-9876",
    date_of_birth="1994-08-15",
    country_code="SE"
)

In [ ]:
# ── Verify: check that the two NEW users landed in the right region ──
# Narrow claim, narrowly checked: these two rows, and only these two rows.

print("🔍 Verifying Residency of the Two New Users")
print("=" * 60)

NEW_USERS = {
    "max.weber@example.de":       "eu-west",   # DE
    "ingrid.svensson@example.se": "eu-north",  # SE
}

actual = {email: set() for email in NEW_USERS}
for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("""
        SELECT email, full_name, country_code, home_region
        FROM users
        WHERE email = ANY(%s)
    """, (list(NEW_USERS),))
    rows = cur.fetchall()

    print(f"\n📦 {region.upper()} database:")
    if rows:
        for email, name, country, home in rows:
            actual[email].add(region)
            print(f"   ✅ Found: {name} ({email}) — country={country}, region={home}")
    else:
        print("   (no matching users — correct! They belong to the other region)")
    conn.close()

# Assert the actual claim, rather than printing a claim we never checked.
for email, expected_region in NEW_USERS.items():
    assert actual[email] == {expected_region}, (
        f"{email} should exist in exactly {{{expected_region}}}, "
        f"but was found in {sorted(actual[email])}"
    )

print("\n💡 Both new users are in exactly one region — the one their country maps to.")
print("   That is geo-routing working. It is NOT yet 'data residency', because")
print("   it only describes rows this notebook wrote. Next cell: everything else.")


### ⚠️ Now scan the *whole* table, not just the rows we just wrote

It is very easy to write a demo that "proves" data residency by querying only
the two records the demo itself created. That proves the demo works. It says
nothing about the other fifteen rows already sitting in the database.

So let's ask the uncomfortable question: for **every** user in **both**
databases, where does that person's data actually live?


In [ ]:
# ── Full-table residency scan ───────────────────────────────
# For every user in every region, record which physical databases hold them.

from collections import defaultdict

def scan_all_residency():
    """email -> {home_region_label, regions the row is physically in}"""
    found = defaultdict(lambda: {"home_region": None, "country": None, "in_regions": set()})
    for region in ["eu-west", "eu-north"]:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("SELECT email, country_code, home_region FROM users")
        for email, country, home in cur.fetchall():
            rec = found[email]
            rec["home_region"] = home
            rec["country"] = country
            rec["in_regions"].add(region)
        conn.close()
    return found

residency = scan_all_residency()

single_region = {e: r for e, r in residency.items() if len(r["in_regions"]) == 1}
both_regions  = {e: r for e, r in residency.items() if len(r["in_regions"]) == 2}

print("🌍 FULL RESIDENCY SCAN")
print("=" * 60)
print(f"   Distinct people          : {len(residency)}")
print(f"   In exactly one region    : {len(single_region)}")
print(f"   In BOTH regions          : {len(both_regions)}")

print("\n   In both regions (first 5):")
for email in sorted(both_regions)[:5]:
    r = both_regions[email]
    print(f"     {email:<34} country={r['country']}  home={r['home_region']}")

# This is the point of the cell.  docker-compose loads the SAME init.sql into
# BOTH containers, so every seeded user starts life in both databases.  That is
# not a bug in the lab — it is the *baseline* the replication notebook needs.
# But it means "each user exists only in their home region" is FALSE here, and
# any demo that claims otherwise while only querying its own two rows is
# telling you something it did not check.
assert both_regions, (
    "expected the seed data to exist in BOTH regions (init.sql is loaded into "
    "both containers). If this fails, the seed baseline changed and the "
    "replication notebook's starting assumptions are wrong."
)

print("\n💡 Read that carefully:")
print("   • Geo-routing decides where a NEW write goes. ✅ demonstrated above.")
print("   • Data residency is a property of the WHOLE dataset over time, including")
print("     seeds, replicas, backups, exports, caches and logs. ❌ not yet shown.")
print("   • Both databases here are notionally in the EU, so holding a Dutch user")
print("     in Ireland is not a Chapter V transfer. It IS a fact you must be able")
print("     to state on demand — that is what Notebook 4 audits.")


## 4. Why Does Microsoft Use This Pattern?

Microsoft Azure is one of the biggest cloud providers in the world. They serve EU governments, banks, hospitals, and enterprises that **legally cannot** store data outside the EU.

### The Business Case (stated carefully)

- **EU public-sector procurement** routinely *contracts for* data residency — the requirement comes from the tender document, not from GDPR.
- **German supervisory authorities** are among the most assertive in Europe, and German professional-secrecy law (§203 StGB) constrains who may process health, legal and social data. There is still no blanket "all German data stays in Germany" statute.
- **PSD2 does not contain a data-localisation rule.** What financial firms actually face is EBA/DORA outsourcing and ICT-risk requirements: know where processing happens, keep audit and regulator access rights, be able to exit. That drives in-region deployments without ever mandating them.
- **There is no EU HIPAA.** Health data is an Article 9 special category (extra conditions to process it), and Article 9(4) lets member states add their own conditions — which is where in-country storage requirements actually come from, when they exist at all.

The honest summary: *contracts and national/sector rules* create residency requirements far more often than GDPR does.

### How Azure Supports It

1. **Azure Front Door / Traffic Manager** route requests — note they route on *latency or priority*, not on legal jurisdiction. Jurisdictional routing is application logic. It is the thing we are writing by hand in this notebook.
2. **Azure SQL active geo-replication / failover groups** replicate to a secondary region **you choose** — not necessarily the platform pair.
3. **Azure Policy** with `allowedLocations` denies resource creation outside approved regions. This is the closest thing to an actual enforced boundary.
4. **Microsoft Purview Compliance Manager** produces a *readiness score* against control frameworks. Read it as "controls we have evidence for", not as a compliance verdict — Microsoft says as much in its own documentation.
5. **EU Data Boundary** is the contractual commitment about where EU customer data is stored and processed.

### What We Simulated

In this notebook, `get_region_for_country()` is the **application-level jurisdiction router** — the piece Azure does *not* give you. Traffic Manager and Front Door pick a region for latency and availability; deciding that a Swedish data subject's row belongs in a specific legal jurisdiction is a decision only your domain model can make.

In [ ]:
# ── Check data residency logs ──────────────────────────────

print("📋 Data Residency Audit Log")
print("=" * 80)

for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("""
        SELECT drl.user_id, u.full_name, drl.action, drl.source_region,
               drl.table_name, drl.reason, drl.created_at
        FROM data_residency_log drl
        LEFT JOIN users u ON drl.user_id = u.id
        ORDER BY drl.created_at DESC
        LIMIT 5
    """)

    print(f"\n📦 {region.upper()} — Recent Residency Events:")
    for row in cur.fetchall():
        print(f"   [{row[6]}] User {row[1] or row[0]}: {row[2]} in {row[3]} ({row[4]})")
        print(f"     Reason: {row[5]}")
    conn.close()

print("\n💡 These logs are essential for GDPR compliance audits.")
print("   They prove that data was stored in the correct region.")

## 5. What Happens with Non-EU Users?

First, correct the premise. **GDPR does not apply to "EU citizens".** Article 3 gives it two triggers:

- **Article 3(1)** — the controller or processor has an *establishment in the Union*, and processing happens in the context of that establishment. Applies regardless of where the data subject is.
- **Article 3(2)** — the controller is outside the Union, but processes data of data subjects *who are in the Union*, either to offer them goods/services or to monitor their behaviour.

So nationality is irrelevant. A German citizen who has emigrated to Brazil and buys from a Brazilian shop with no EU presence is not covered; a Brazilian tourist in Lisbon buying from that same shop's EU subsidiary is. Your `country_code` column is a proxy for *where the person is*, and a rough one — which is itself worth knowing.

In a real system, you'd have additional region mappings:
- US users → US East + US West paired regions
- Asian users → East Asia + Southeast Asia paired regions
- etc.

The key insight: **the architecture is the same everywhere** — pair regions within the same legal jurisdiction.

In [ ]:
# ── Demo: What happens with an unmapped country? ───────────

try:
    region = get_region_for_country("US")
except ValueError as e:
    print(f"⚠️  {e}")
    print()
    print("In a real system, you would:")
    print("1. Route US users to a US-based region pair (e.g. East US + West US)")
    print("2. Apply US state privacy law (CCPA/CPRA, VCDPA, ...) — there is no")
    print("   single federal equivalent of GDPR")
    print("3. Decide deliberately whether EU and US data share infrastructure.")
    print("   Mixing them is not automatically unlawful — it means every EU row")
    print("   that reaches US infrastructure is a Chapter V transfer you must")
    print("   have a mechanism and a record for.")

# The unmapped-country path must actually raise. If someone later adds a
# catch-all default region, this demo would silently start "succeeding" and
# stop teaching anything.
for unmapped in ("US", "BR", "JP"):
    try:
        get_region_for_country(unmapped)
    except ValueError:
        pass
    else:
        raise AssertionError(
            f"get_region_for_country({unmapped!r}) returned a region instead of "
            f"refusing — an unmapped country must never silently get a default"
        )

# Switzerland is the interesting one: close to the EU, covered by adequacy,
# but NOT in the EU/EEA. It must be refused with the adequacy explanation.
try:
    get_region_for_country("CH")
except ValueError as e:
    assert "adequacy" in str(e).lower(), f"unexpected message for CH: {e}"
    print(f"\n🇨🇭 {e}")

## 🎯 Key Takeaways

1. **Personal data is broader than "PII"** — Article 4(1) covers anything *relating to* an identifiable person, including data that only identifies in combination with something else.
2. **GDPR is not a localisation law.** It restricts *transfers* to third countries (Chapter V). Keeping data in-region is an engineering strategy that avoids that work; real localisation mandates come from contracts, national law, and sector regulators.
3. **GDPR applies by territory, not nationality** — Article 3 is about establishments in the Union and data subjects located in the Union.
4. **Geo-routing** sends writes to the correct regional database based on the user's jurisdiction. Azure's traffic routers do *not* do this for you; it is application logic.
5. **Azure Paired Regions are a reliability feature that is usually geography-aware.** "Almost all" pairs stay in-geography (Brazil South ↔ South Central US is the documented exception), only a small number of services use pairs at all, and pairing alone gives you no automatic failover.
6. **A residency claim you have not queried is not a claim.** Verifying the two rows your demo just wrote proves nothing about the other fifteen — as the full-table scan showed.
7. **Audit logs are evidence, not compliance** — log where data was written and why, so you can answer Article 30 questions.

## ⏭️ Next Up

In **Notebook 2**, we'll replicate between the pair for disaster recovery — and, more usefully, watch the asynchronous replication window actually lose a write when the primary dies at the wrong moment.